# Lab 01 — Neural computation: perceptron → Adaline → nonlinear model

**Dataset:** [ylecun/mnist](https://huggingface.co/datasets/ylecun/mnist), a real handwritten-digit dataset. We restrict the task to digits **0 vs 1** so the first principles remain visible.


> **Experiment contract:** do not change the intervention before writing your prediction. Every run reports the same metric and budget so the comparison is interpretable.


### What you will produce
A baseline table comparing a perceptron and Adaline, a learning-rate intervention, a shuffled-label control, and a short conclusion.

## The math behind the experiment

A **perceptron** computes a weighted score and then applies a hard decision:

$$
z = \mathbf{w}^{\top}\mathbf{x}+b,\qquad \hat{y}=\mathbf{1}[z\ge 0].
$$

**Adaline** means *Adaptive Linear Neuron*. Unlike the perceptron, it does **not** train on the hard 0/1 decision. It trains the continuous score $z$ and minimizes mean-squared error:

$$
L=\frac{1}{N}\sum_{i=1}^{N}(y_i-z_i)^2.
$$

That difference explains why learning rate matters: the gradient is proportional to the continuous prediction error.

After training, we can still turn the score into a class using a threshold. The notebook's experiment therefore compares two different **learning mechanisms**, not merely two names for the same classifier.

In [ ]:
!pip -q install datasets scikit-learn pandas matplotlib
from datasets import load_dataset
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

seed=7
train=load_dataset("ylecun/mnist", split="train[:5000]")
test=load_dataset("ylecun/mnist", split="test[:1000]")
def xy(ds):
    X=np.array([np.asarray(x).reshape(-1) for x in ds["image"]],dtype=np.float32)/255.0
    y=np.array(ds["label"])
    m=(y==0)|(y==1)
    return X[m],y[m]
Xtr,ytr=xy(train); Xte,yte=xy(test)
print(Xtr.shape, Xte.shape, "positive rate", ytr.mean())

## Step 1 — Predict

Before running training, record:

- A perceptron should perform strongly because 0/1 have substantial but not perfect linear structure after flattening.
- Adaline should be sensitive to feature scaling and learning rate because it optimizes a continuous loss.
- Shuffling labels should collapse test accuracy toward chance.

The last cell contains the answer key; do not read it until you have written your prediction.

In [ ]:
class Perceptron:
    def __init__(self,lr=0.01,epochs=10): self.lr,self.epochs=lr,epochs
    def fit(self,X,y):
        self.w=np.zeros(X.shape[1]); self.b=0.
        for _ in range(self.epochs):
            for xi,yi in zip(X,y):
                pred=1 if xi@self.w+self.b>=0 else 0
                err=yi-pred; self.w += self.lr*err*xi; self.b += self.lr*err
        return self
    def predict(self,X): return ((X@self.w+self.b)>=0).astype(int)

class Adaline:
    def __init__(self,lr=0.05,epochs=15): self.lr,self.epochs=lr,epochs
    def fit(self,X,y):
        self.w=np.zeros(X.shape[1]); self.b=0.
        for _ in range(self.epochs):
            z=X@self.w+self.b; e=z-y
            self.w -= self.lr*(2/len(X))*X.T@e; self.b -= self.lr*2*e.mean()
        return self
    def predict(self,X): return ((X@self.w+self.b)>=0.5).astype(int)

runs=[]
for name,model in [("Perceptron",Perceptron()),("Adaline",Adaline())]:
    model.fit(Xtr,ytr); runs.append([name,accuracy_score(ytr,model.predict(Xtr)),accuracy_score(yte,model.predict(Xte))])
pd.DataFrame(runs,columns=["model","train_accuracy","test_accuracy"])

## Step 2 — Controlled intervention: learning rate

Change only the Adaline learning rate. Keep the data, epochs, initialization and test set fixed. A fair experiment changes one causal factor at a time.

In [ ]:
rows=[]
for lr in [0.005,0.02,0.05,0.2]:
    m=Adaline(lr=lr,epochs=15).fit(Xtr,ytr)
    rows.append(["Adaline",lr,accuracy_score(ytr,m.predict(Xtr)),accuracy_score(yte,m.predict(Xte))])
rate_table=pd.DataFrame(rows,columns=["model","learning_rate","train_accuracy","test_accuracy"])
rate_table

## Step 3 — Failure control: shuffled labels

If the model still appears excellent after labels are permuted, investigate leakage or an evaluation bug. A capable model can memorize training labels, but it should not retain genuine test performance when the correspondence between inputs and labels is destroyed.

In [ ]:
rng=np.random.default_rng(seed)
ysh=rng.permutation(ytr)
m=Perceptron(lr=0.01,epochs=10).fit(Xtr,ysh)
print("Shuffled-label train:",accuracy_score(ysh,m.predict(Xtr)))
print("Shuffled-label test :",accuracy_score(yte,m.predict(Xte)))
summary=pd.DataFrame(runs,columns=["model","train_accuracy","test_accuracy"])
print("\nBASELINE RESULTS"); print(summary.to_string(index=False))
print("\nINTERVENTION RESULTS"); print(rate_table.to_string(index=False))

## Read the graphs — optimization is more than one accuracy number

The first graph compares training and held-out accuracy for the baseline models. A large gap can signal overfitting; low values on both sets suggest underfitting or a failed optimization setup. The second graph changes only Adaline's learning rate, making the effect of the step size visible.

These figures are generated from this notebook's actual run. Do not expect exact values to match across every environment; interpret the pattern and inspect the shuffled-label control.

In [ ]:
import matplotlib.pyplot as plt

baseline_plot = pd.DataFrame(runs, columns=["model", "train_accuracy", "test_accuracy"])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
baseline_plot.set_index("model")[["train_accuracy", "test_accuracy"]].plot(
    kind="bar", ax=axes[0], ylim=(0, 1), rot=0
)
axes[0].set_title("Training vs held-out accuracy")
axes[0].set_ylabel("Accuracy")
axes[0].set_xlabel("")
axes[0].grid(axis="y", alpha=.25)

axes[1].plot(rate_table["learning_rate"], rate_table["train_accuracy"], marker="o", label="Train")
axes[1].plot(rate_table["learning_rate"], rate_table["test_accuracy"], marker="s", label="Test")
axes[1].set_xscale("log")
axes[1].set_ylim(0, 1)
axes[1].set_title("Adaline: learning-rate sensitivity")
axes[1].set_xlabel("Learning rate (log scale)")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(alpha=.25)
fig.tight_layout()
plt.show()

## Conclusion / answer key

**Expected pattern:** the original-label models should beat the shuffled-label control on held-out data; Adaline should show a learning-rate regime where optimization is stable and another regime where updates become unstable or less reliable. The exact numbers depend on the sampled subset and runtime environment.

### Questions
1. What evidence distinguishes learning from memorization? **Held-out performance plus the shuffled-label control.**
2. Why is Adaline more learning-rate sensitive? **Its gradient update scales with the continuous prediction error.**
3. What should you investigate if shuffled labels remain highly accurate on test data? **A data/evaluation leakage bug first.**

### Research extension
Repeat each condition for three seeds and report mean ± standard deviation.

### Reproducibility
Record Python/PyTorch/sklearn versions, dataset revision, subset sizes, seed, CPU/GPU, and the exact code revision. Do not report only the best run.